### Load libraries and and engine

In [1]:
#using Pkg
#Pkg.add(["Revise", "MLDatasets"]) # Run once
using Revise
using Random
using MLDatasets
using Statistics

includet("engines/cnn_library_v5.jl")

### Config

In [ ]:
Base.@kwdef mutable struct Config  # @kdef allows convenient use of Config() constructor e.g. cfg = Config(rand_seed=4, lr=1e-1)
    lr::Float32 = 1f-2
    num_epochs::Int = 3
    batch_size::Int = 10
    dropout_p::Float32 = 0.4f0
    rand_seed::Int = 0
end

cfg = Config()
Random.seed!(cfg.rand_seed)

TaskLocalRNG()

### Load data

In [ ]:
println("[x] Loading FashionMNIST data...")
train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

x_train = Float32.(reshape(train_data.features, size(train_data.features,1), size(train_data.features,2), 1, :))
x_test  = Float32.(reshape(test_data.features,  size(test_data.features,1),  size(test_data.features,2),  1, :))

y_train = zeros(Float32, 10, size(x_train, 4))
y_test = zeros(Float32, 10, size(x_test, 4))

for (i, class) in enumerate(train_data.targets) y_train[class+1, i] = 1 end
for (i, class) in enumerate(test_data.targets) y_test[class+1, i] = 1 end
println("[+] Data is ready! Training size: ", size(x_train, 4))

[x] Loading FashionMNIST data...
[+] Data is ready! Training size: 60000


### Utils functions

In [ ]:
function test(model, x_test, y_test, input, target, output)
    correct_pred = 0
    N = size(x_test, 4)
    for i in 1:N
        set_eval_mode!(model)
        forward!(model, input => x_test[:, :, :, i], target => y_test[:, i])
        set_train_mode!(model, cfg.dropout_p)
        if argmax(output.data) == argmax(y_test[:, i])
            correct_pred += 1
        end
    end
    acc = correct_pred / N
    println("Accuracy: $(round(acc * 100, digits=2))%")
    return acc
end

function train!(model, x_train, y_train, opt, input, target, batchsize)
    N = size(x_train, 4)
    indices = shuffle(1:N)
    total_loss = 0.0f0
    for batch_start in 1:batchsize:N
        batch_end = min(batch_start + batchsize - 1, N)
        for sample in indices[batch_start:batch_end]
            zerograd!(model)
            forward!(model, input => x_train[:, :, :, sample], target => y_train[:, sample])
            backward!(model)
            accumulate!(opt, model)
            total_loss += model[end].data[1]
        end
        optimize!(opt, model)
    end
    return total_loss
end

function set_train_mode!(graph, p)
    for node in graph
        if node isa GraphNode{:dropout, 3}
            node.args[2].data[1] = p
        end
    end
end

function set_eval_mode!(graph)
    for node in graph
        if node isa GraphNode{:dropout, 3}
            node.args[2].data[1] = 0.0f0
        end
    end
end

set_eval_mode! (generic function with 1 method)

### Define model

In [ ]:
net = chain((
    conv((3,3), 1 => 6, 1),
    maxPool((2, 2)),
    conv((3,3), 6 => 16, 1),
    maxPool((2, 2)),
    flatten(),
    dense(784 => 84, relu),
    dropout(cfg.dropout_p),
    dense(84 => 10),
))

input = tensor(28, 28, 1)
target = tensor(10)
target.data[1] = 1f0
output = net(input)
loss = lce(output, target)
model = graph(loss)
opt = GradientDescent(cfg.lr)

GradientDescent(Dict{GraphWeight, Array{Float64}}(), 0.01, 0)

### Train and test on full dataset

In [6]:
print("[x] Accuracy before training (Random model): ")
test(model, x_test, y_test, input, target, output)

println("\n[x] Training started...")
for epoch in 1:cfg.num_epochs
    t = @elapsed L = train!(model, x_train, y_train, opt, input, target, cfg.batch_size)
    println("[+] Epoch $epoch | Loss: $(L) | Time: $(round(t, digits=2))s")
end

print("\n[x] Accuracy after training (Final model): ")
test(model, x_test, y_test, input, target, output)

[x] Accuracy before training (Random model): Accuracy: 13.05%

[x] Training started...
[+] Epoch 1 | Loss: 33241.237963496 | Time: 37.74s
[+] Epoch 2 | Loss: 22653.227158413727 | Time: 26.55s
[+] Epoch 3 | Loss: 19914.453214444362 | Time: 26.51s

[x] Accuracy after training (Final model): Accuracy: 87.16%


0.8716